In [ ]:
import pandas as pd
manual_df = pd.read_csv("20240524-0607_RGCB_MVC.csv")
auto_df = pd.read_csv("buowset_preds_96545a6b81b24794962d464e8fcbdc1d.csv")

In [ ]:
manual_df["Begin Path"]

In [ ]:
manual_df["file_key"] = manual_df["Begin Path"].str.split("Y:/Audiomoth/Raw sound files/").apply(lambda x: x[1])

In [ ]:
len(manual_df["file_key"].unique())

In [ ]:
auto_df["file_key"] = auto_df["file_path"].str.split("/mnt/restorage/Audiomoth/Raw sound files/").apply(lambda x: x[1])

In [ ]:
len(auto_df["file_key"].unique()) 

In [ ]:
auto_df_filtered = auto_df[auto_df["file_key"].isin(manual_df["file_key"].unique())]

In [ ]:
auto_df_filtered

In [ ]:
def soft_chunking(file_df):
    rows = []
    for i in range(
            0, 
            int(file_df["End Time (s)"].max()//3) * 3 + 9, 
            3):
        anno_df = file_df[~((file_df["End Time (s)"] < i)|(file_df["Begin Time (s)"] > i + 3))]
        if anno_df.shape[0] == 0:
            continue
        

        anno_df["offset"] = i  
        rows.append(anno_df[["file_key", "Call Type", "offset"]])
    return pd.concat(rows).drop_duplicates()


manual_df_chunked = manual_df.groupby(by="Begin Path").apply(soft_chunking).reset_index(drop=True)
manual_df_chunked

/tmp/ipykernel_2298686/2604827580.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  manual_df.groupby(by="Begin Path").apply(soft_chunking)


file_key  \
Begin Path                                                                                                  
Y:/Audiomoth/Raw sound files/2024/RGCB/RGCB01/B... 4    2024/RGCB/RGCB01/B017/20240524-0607_017_period...   
                                                   6    2024/RGCB/RGCB01/B017/20240524-0607_017_period...   
                                                   0    2024/RGCB/RGCB01/B017/20240524-0607_017_period...   
                                                   1    2024/RGCB/RGCB01/B017/20240524-0607_017_period...   
                                                   11   2024/RGCB/RGCB01/B017/20240524-0607_017_period...   
...                                                                                                   ...   
Y:/Audiomoth/Raw sound files/2024/RGCB/RGCB08/B... 705  2024/RGCB/RGCB08/B021/20240524-0607_021_period...   
                                                   705  2024/RGCB/RGCB08/B021/20240524-0607_021_period...   
                                                   661  2024/RGCB/RGCB08/B021/20240524-0607_021_period...   
                                                   661  2024/RGCB/RGCB08/B021/20240524-0607_021_period...   
Y:/Audiomoth/Raw sound files/2024/RGCB/RGCB08/B... 707  2024/RGCB/RGCB08/B021/20240524-0607_021_period...   

                                                       Call Type  offset  
Begin Path                                                                
Y:/Audiomoth/Raw sound files/2024/RGCB/RGCB01/B... 4       Alarm     945  
                                                   6       Alarm     948  
                                                   0       Alarm     951  
                                                   1       Alarm     954  
                                                   11      Alarm     957  
...                                                          ...     ...  
Y:/Audiomoth/Raw sound files/2024/RGCB/RGCB08/B... 705     Alarm     924  
                                                   705     Alarm     927  
                                                   661     Alarm    1284  
                                                   661     Alarm    1287  
Y:/Audiomoth/Raw sound files/2024/RGCB/RGCB08/B... 707     Alarm    1299  

[659 rows x 3 columns]

In [ ]:
manual_df_chunked